[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day22-load-testing-slos.ipynb)

# Day 22 — Load Testing + SLOs
**Time:** 30–60 min · **Prereqs:** Day 21 (TTFT, TPOT, goodput, bench.py) · **You need:** CPU only — this lab is an event-driven simulation, no GPU.

**Today you'll be able to**
1. Write numeric SLOs for a chat product and justify each number from human perception and error budgets.
2. Generate Poisson arrivals + mixed-length workloads, and show with numbers why fixed-concurrency tests lie.
3. Compute offered load in Erlangs and pick capacity for a target utilization.
4. Run a load test, find R_max, and name what breaks first — queue, KV memory, or preemptions.

In [ ]:
# Cell 2: installs. numpy/matplotlib are preinstalled on Colab;
# this is a no-op guard so the notebook runs anywhere.
import importlib, subprocess, sys
for pkg in ("numpy", "matplotlib"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("deps ready")

# Expected output:
# deps ready


## 1. The simulator: a mock serving system

Instead of a GPU server we simulate one — the queueing dynamics are the lesson,
and a simulator lets us run 3 virtual minutes per load level in under a second.
The model (all parameters are real serving numbers, not toys):

| Parameter | Value | Basis |
|---|---|---|
| Arrival process | Poisson (exponential gaps) | real traffic |
| Prompt mix | 80% 128–512 tok, 20% 1000–2000 tok | chat + RAG-ish |
| Output mix | 70% 20–80 tok, 30% 100–300 tok | chat |
| Decode slots | 12 concurrent | small GPU server |
| Prefill rate | 3000 tok/s | 70B-class prefill |
| TPOT base | 20 ms, mild contention growth | H100-class decode |
| KV per token | 320 KB | 70B GQA fp8 (Day 8 math) |
| KV budget | 8 GB | generous — the sweep studies queueing |
| SLOs | TTFT p99 < 1 s, TPOT p99 < 100 ms | Day 22 §3.1 |

**Mechanisms:** FIFO queue; prefill + decode occupy a slot; per-token TPOT drawn
lognormal around a contention-scaled base; when KV would overflow, the **newest
request still in prefill is preempted** (vLLM's newest-first policy) and
**swapped to CPU — it resumes, never recomputes from scratch**.

**Documented approximation:** real vLLM preempts when a *running* request needs
another KV block mid-decode; we check at admission instants with max-token
reservation. Victim policy and swap-resume are faithful; only the check timing
is simplified.

In [ ]:
# Cell 4: the simulator (event-driven, request-level)
"""Day 22 simulator: event-driven mock serving system.
FIFO queue, SLOTS decode slots, KV budget with vLLM-style preemption of the
newest in-prefill request, per-token lognormal TPOT jitter.
"""
import heapq
import math
from collections import deque

import numpy as np

PREFILL_RATE = 3000.0      # tok/s
TPOT_BASE = 0.020          # s per token, unloaded
SLOTS = 12                 # concurrent decode slots
KV_PER_TOKEN = 320 * 1024  # bytes (70B-class: 2*80 layers * 8 KV heads * 128 dim * 1B fp8-ish)
KV_BUDGET = 8 * 1024 ** 3  # 8 GB (generous: sweep studies queueing, not KV)
SLO_TTFT = 1.0             # s
SLO_TPOT_P99 = 0.100       # s


def percentile(xs, p):
    xs = np.sort(np.asarray(xs, dtype=float))
    if len(xs) == 0:
        return float("nan")
    k = (len(xs) - 1) * p / 100.0
    f = int(math.floor(k))
    c = min(f + 1, len(xs) - 1)
    return float(xs[f] + (xs[c] - xs[f]) * (k - f))


def sample_prompt_len(n, r):
    pick = r.random(n) < 0.8
    # long prompts capped at 2000 tok (prefill <= 0.67 s) so the 1 s TTFT SLO
    # is achievable at low load -- the SLO story stays about queueing, not prompts
    return np.where(pick, r.uniform(128, 512, n), r.uniform(1000, 2000, n)).astype(int)


def sample_out_len(n, r):
    pick = r.random(n) < 0.7
    return np.where(pick, r.uniform(20, 80, n), r.uniform(100, 300, n)).astype(int)


def run_load(rate, duration=180.0, seed=0, arrival="poisson", kv_budget=KV_BUDGET,
             warmup=30.0):
    r = np.random.default_rng(seed)
    max_n = int(rate * duration * 3) + 16
    if arrival == "poisson":
        gaps = r.exponential(1.0 / rate, max_n)
    elif arrival == "fixed":
        gaps = np.full(max_n, 1.0 / rate)
    else:
        raise ValueError(arrival)
    arr = np.cumsum(gaps)
    arr = arr[arr < duration]
    n = len(arr)
    prompt = sample_prompt_len(n, r)
    out = sample_out_len(n, r)

    running = []        # dicts: idx, prefill_end, end, kv, admit_t, stale
    waiting = deque()   # indices, FIFO
    comp = []           # heap of (end_time, idx, reqdict)
    rem = [None] * n    # remaining prefill (s) if preempted mid-prefill (swap-resume)
    gaps_cache = [None] * n  # per-request token gaps, drawn once
    res = [None] * n
    kv_used = 0
    preemptions = 0
    max_q = 0
    active_area = 0.0
    now = 0.0
    svc_sum = 0.0   # sum of (end - admit) over every admission (partial included)
    admits = 0

    def do_admit(i, t):
        """Try to admit request i at time t. Returns (ok, victims)."""
        nonlocal kv_used, preemptions, svc_sum, admits
        need = (int(prompt[i]) + int(out[i])) * KV_PER_TOKEN
        victims = []
        # NOTE (approximation, documented in notebook): real vLLM preempts when a
        # *running* request needs another KV block mid-decode; we check at admission
        # instants with max-token reservation instead. Victim policy (newest still
        # in prefill) matches vLLM; evicted requests swap to CPU and resume.
        while running and kv_used + need > kv_budget:
            cands = [q for q in running
                     if t < q["prefill_end"] and q["admit_t"] < t]
            if not cands:
                break
            victim = max(cands, key=lambda q: q["admit_t"])
            running.remove(victim)
            victim["stale"] = True
            kv_used -= victim["kv"]
            preemptions += 1
            j = victim["idx"]
            # swap-to-CPU: preserve prefill progress, resume on re-admit
            rem[j] = max(0.0, victim["prefill_end"] - t)
            victims.append(j)
        if len(running) >= SLOTS or kv_used + need > kv_budget:
            return False, victims
        pre = rem[i] if rem[i] is not None else int(prompt[i]) / PREFILL_RATE
        rem[i] = None
        decode_start = t + pre
        if gaps_cache[i] is None:
            tpot_eff = TPOT_BASE * max(1.0, len(running) / 10.0)
            gaps_cache[i] = tpot_eff * r.lognormal(0.0, 0.15, int(out[i]))
        gg = gaps_cache[i]
        end = decode_start + float(gg.sum())
        ttft = (decode_start - arr[i]) + float(gg[0])
        tp99 = percentile(gg, 99)
        res[i] = dict(ttft=ttft, tpot_mean=float(gg.mean()), tpot_p99=tp99,
                      e2e=end - arr[i], out=int(out[i]),
                      attains=(ttft < SLO_TTFT and tp99 < SLO_TPOT_P99),
                      arr=float(arr[i]), service=end - t)
        svc_sum += end - t
        admits += 1
        kv_used += need
        q = dict(idx=i, prefill_end=decode_start, end=end, kv=need,
                 admit_t=t, stale=False)
        running.append(q)
        heapq.heappush(comp, (end, i, q))
        return True, victims

    def do_finish(t):
        nonlocal kv_used
        while comp and comp[0][0] <= t:
            _, _, q = heapq.heappop(comp)
            if q["stale"]:
                continue
            if q in running:
                running.remove(q)
                kv_used -= q["kv"]

    def drain_waiting(t, frozen):
        while waiting:
            j = waiting[0]
            if j in frozen:
                break  # preempted at this t: retry at the next event, not now
            waiting.popleft()
            ok, victims = do_admit(j, t)
            frozen.update(victims)
            for v in victims:
                waiting.appendleft(v)
            if not ok:
                waiting.appendleft(j)
                break

    i = 0
    INF = float("inf")
    while i < n or comp or waiting:
        t_arr = float(arr[i]) if i < n else INF
        t_cmp = comp[0][0] if comp else INF
        t = min(t_arr, t_cmp)
        active_area += len(running) * (t - now)
        now = t
        do_finish(t)
        frozen = set()  # shared: arrivals + drain at this event time
        while i < n and float(arr[i]) <= t:
            ok, victims = do_admit(i, t)
            frozen.update(victims)
            for v in victims:
                waiting.appendleft(v)
            if not ok:
                waiting.append(i)
            i += 1
        drain_waiting(t, frozen)
        max_q = max(max_q, len(waiting))

    meas = [res[k] for k in range(n) if res[k] is not None and res[k]["arr"] >= warmup]
    ttfts = [m["ttft"] for m in meas]
    tp99s = [m["tpot_p99"] for m in meas]
    tmeans = [m["tpot_mean"] for m in meas]
    toks = sum(m["out"] for m in meas)
    good = sum(m["out"] for m in meas if m["attains"])
    wall = duration - warmup
    mean_service = float(np.mean([m["service"] for m in meas])) if meas else 0.0
    area_now = active_area / max(1e-9, now)
    return dict(
        n_meas=len(meas), rate=rate,
        attainment=sum(m["attains"] for m in meas) / max(1, len(meas)),
        ttft_p50=percentile(ttfts, 50), ttft_p99=percentile(ttfts, 99),
        tpot_p99_of_means=percentile(tmeans, 99),
        tpot_p99_p99=percentile(tp99s, 99),
        throughput=toks / wall, goodput=good / wall,
        preemptions=preemptions, max_queue=max_q,
        little_running=area_now,
        # exact accounting identity: integrated active-time == sum of services
        area_check=abs(active_area - svc_sum) / max(1e-9, svc_sum),
        little_theory=rate * svc_sum / max(1, admits),
        little_gap=abs(area_now - rate * svc_sum / max(1, admits)) / max(1e-9, rate * svc_sum / max(1, admits)),
        mean_service=mean_service,
    )

# Expected output: (no output -- definitions only)


## 2. Calibration: trust, then verify

Two checks before any load test. First, a single request on an empty server:
TTFT should be ~prefill + one decode step (~0.1–0.2 s for a short prompt).
Second, **Little's law**: time-average active requests must equal
arrival rate × mean service time. The simulator also asserts its own
bookkeeping identity: integrated active-time == sum of all services.

In [ ]:
# Cell 6: calibration
c = run_load(rate=0.5, duration=600.0, seed=1)
print(f"mean_service = {c['mean_service']:.2f} s   TTFT p50 = {c['ttft_p50']:.2f} s")
print(f"accounting rel err = {c['area_check']:.1e}  (must be ~0)")
print(f"Little: running_avg = {c['little_running']:.3f}, "
      f"theory = {c['little_theory']:.3f}  (gap {c['little_gap']*100:.1f}%)")
assert c["area_check"] < 1e-9
assert c["little_gap"] < 0.05
assert 0.05 < c["ttft_p50"] < 0.6
ES_NOLOAD = c["mean_service"]  # no-load mean service: the E[S] for Erlang math
print("calibration OK: the simulator conserves request-time and obeys Little's law")

# Expected output (seeds are fixed, so these are exact):
# mean_service = 1.94 s   TTFT p50 = 0.14 s
# accounting rel err = 1.9e-16  (must be ~0)
# Little: running_avg = 0.976, theory = 0.971  (gap 0.5%)
# calibration OK: the simulator conserves request-time and obeys Little's law


## 3. The load sweep: find R_max

Sweep arrival rate R ∈ {1, 2, 3, 4, 8} req/s, 3 virtual minutes each (first
30 s discarded as warmup). For each R we report SLO attainment (fraction of
requests with TTFT < 1 s **and** TPOT p99 < 100 ms), the tails, throughput,
**goodput** (Day 21: only SLO-meeting tokens count), preemptions, max queue
depth — plus offered load in Erlangs (a = R × E[S]) and utilization
ρ = a / 12 slots.

In [ ]:
# Cell 8: the sweep (Erlangs use the no-load E[S]: offered load, not carried)
print(f"{'R':>3} {'n':>4} {'attain':>7} {'TTFTp99':>8} {'TPOTp99':>8} "
      f"{'thr':>6} {'goodput':>8} {'preempt':>8} {'maxq':>5} {'Erl':>5} {'rho':>5}")
for R in (1, 2, 3, 4, 8):
    s = run_load(rate=R, duration=180.0, seed=7)
    erl = R * ES_NOLOAD
    print(f"{R:>3} {s['n_meas']:>4} {s['attainment']*100:>6.1f}% "
          f"{s['ttft_p99']:>7.2f}s {s['tpot_p99_p99']*1000:>7.1f}ms "
          f"{s['throughput']:>6.0f} {s['goodput']:>8.0f} "
          f"{s['preemptions']:>8} {s['max_queue']:>5} {erl:>5.1f} {erl/12:>5.2f}")

# Expected output (seeds fixed):
#   R    n  attain  TTFTp99  TPOTp99    thr  goodput  preempt  maxq   Erl   rho
#   1  136  100.0%    0.64s    32.1ms     91       91        0     0   1.9  0.16
#   2  293  100.0%    0.68s    30.9ms    202      202        0     0   3.9  0.32
#   3  430  100.0%    0.66s    31.3ms    267      267        0     1   5.8  0.49
#   4  609   92.8%    2.78s    33.2ms    392      364        0    11   7.8  0.65
#   8 1205    0.0%  101.41s    34.3ms    772        0        0   527  15.5  1.29
#
# Read it: R_max ~= 3 req/s (last R with ~100% attainment). At R=8 the server
# still pushes 772 tok/s of raw throughput -- and ZERO goodput. Every token
# burned bought no satisfied user. That is the Day 21 knee, located by
# arrival rate instead of concurrency.


## 4. R_max: reading the knee

Plot attainment and goodput against R. Two curves every load test produces:
attainment falls off a cliff past R_max, and goodput peaks *at* R_max then
collapses while raw throughput keeps climbing — the server looks busy and
achieves nothing.

In [ ]:
# Cell 10: plot the knee
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Rs, att, good, thr = [], [], [], []
for R in (1, 2, 3, 4, 6, 8):
    s = run_load(rate=R, duration=180.0, seed=7)
    Rs.append(R); att.append(100 * s["attainment"])
    good.append(s["goodput"]); thr.append(s["throughput"])

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(Rs, att, "o-", label="SLO attainment %")
ax1.axhline(99, color="gray", ls="--", lw=1, label="99% SLO bar")
ax1.set_xlabel("arrival rate R (req/s)"); ax1.set_ylabel("attainment %")
ax1.set_ylim(-5, 105)
ax2 = ax1.twinx()
ax2.plot(Rs, good, "s--", color="tab:orange", label="goodput tok/s")
ax2.plot(Rs, thr, "x:", color="tab:red", label="raw throughput tok/s")
ax2.set_ylabel("tok/s")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right", fontsize=8)
plt.title("Day 22: the knee -- R_max ~= 3 req/s")
plt.tight_layout(); plt.savefig("day22-knee.png", dpi=100)
print("saved day22-knee.png -- attainment cliff just past R=3; "
      "goodput peaks at R=3 while raw throughput keeps rising")

# Expected output:
# saved day22-knee.png -- attainment cliff just past R=3; goodput peaks at R=3
# while raw throughput keeps rising


## 5. Why Poisson, not fixed concurrency

Same mean rate (4 req/s), two arrival patterns: metronome spacing vs Poisson.
The curriculum's worked example: at λ=5/s, ~8% of 200 ms windows see ≥3× the
mean arrivals. Fixed-concurrency tests never show the server that world.

In [ ]:
# Cell 12: Poisson vs fixed spacing at matched mean rate
for mode in ("fixed", "poisson"):
    s = run_load(rate=4, duration=180.0, seed=7, arrival=mode)
    print(f"{mode:8s}: attain={s['attainment']*100:5.1f}%  "
          f"TTFT p99={s['ttft_p99']:5.2f}s  maxq={s['max_queue']}")
print("Poisson bursts make the p99 ~4x worse at the SAME mean rate -- "
      "this is what fixed-concurrency benchmarks hide.")

# Expected output (seeds fixed):
# fixed   : attain=100.0%  TTFT p99= 0.65s  maxq=2
# poisson : attain= 92.8%  TTFT p99= 2.78s  maxq=11
# Poisson bursts make the p99 ~4x worse at the SAME mean rate -- this is what
# fixed-concurrency benchmarks hide.


## 6. What breaks first: queue or KV memory?

R=3 is perfectly healthy with a generous 8 GB KV budget. Shrink the budget to
2 GB — same arrival rate, same prompts — and watch which limit binds first.
This is the Day 19/20 mechanism (PagedAttention blocks, newest-first
preemption) made visible in the load test.

In [ ]:
# Cell 14: KV pressure in isolation (R=3, healthy for compute)
import os
for b in (8 * 1024**3, 2 * 1024**3):
    s = run_load(rate=3, duration=180.0, seed=7, kv_budget=b)
    print(f"KV={b/1024**3:.0f}GB: attain={s['attainment']*100:5.1f}%  "
          f"TTFT p99={s['ttft_p99']:5.2f}s  preemptions={s['preemptions']}")

slos = """# SLOs -- mock 70B-class server (Day 22 load test, seed 7)
## SLOs
- TTFT p99 < 1.0 s, TPOT p99 < 100 ms, availability 99.9% (43.2 min/mo budget)
## R_max
- 3 req/s at 100% attainment (offered 5.8 Erlangs, rho 0.49 on 12 slots)
- goodput at R_max: 267 tok/s
## What breaks first
- Queue wait (TTFT p99) under Poisson bursts: rho=0.65 already bites (R=4: 92.8%)
- KV memory binds before compute when the budget is tight: at R=3, shrinking
  KV 8 GB -> 2 GB drops attainment 100% -> 85.8% with 301 newest-first preemptions
- Raw throughput is a liar past the knee: R=8 does 772 tok/s, goodput 0
## Notes
- Simulator: event-driven, Poisson arrivals, mixed lengths, 12 slots,
  prefill 3000 tok/s, TPOT 20 ms base, KV 320 KB/tok, swap-resume preemption.
- Re-run on a T4 with vLLM: python week03/bench.py --arrival poisson --rate R ...
"""
os.makedirs("week04", exist_ok=True)
open("week04/slos.md", "w").write(slos)
print("wrote week04/slos.md")

# Expected output (seeds fixed):
# KV=8GB: attain=100.0%  TTFT p99= 0.66s  preemptions=0
# KV=2GB: attain= 85.8%  TTFT p99= 4.94s  preemptions=301
# wrote week04/slos.md


## 7. Checkpoints

1. **Sensible chat TPOT p99 SLO?** *(answer in the PDF packet)*
2. **Why Poisson arrivals instead of fixed concurrency?** *(answer in the PDF packet)*
3. **ρ = 1.25 means?** *(answer in the PDF packet)*

## 8. Wrap-up

- **R_max ≈ 3 req/s** for this mock server at 100% SLO attainment; goodput peaks there (267 tok/s).
- Past the knee, raw throughput keeps climbing while goodput collapses (R=8: 772 tok/s raw, 0 goodput).
- Poisson bursts make TTFT p99 ~4× worse than fixed spacing at the same mean rate.
- Tightening KV 8 GB → 2 GB at a healthy R=3 drops attainment to 85.8% via 301 preemptions — **name what breaks first, don't guess it**.

**Tomorrow — Day 23, Structured Output (xgrammar):** a byte-level pushdown
automaton masks 127,997 of 128k logits per step to guarantee valid JSON.
You'll measure what that guarantee costs in TPOT.